# rsn_flat for Colab

A handful of standalone files — `rsn.py`, `offline.py`, `policy.py`, `simulation.py`, `je.py`,
`check.py` — with plain imports, pure Python (no compilers, no installs).

Two ways to use it:

1. **This notebook**: `Runtime → Run all`. Part 1 writes each file
   (every cell is independently copy-pastable), Part 2 runs each one on
   simulation data.
2. **A Drive folder**: save the files into e.g. `My Drive/rsn_flat`, then in any notebook:

```python
from google.colab import drive; drive.mount('/content/drive')
import sys; sys.path.append('/content/drive/My Drive/rsn_flat')
from rsn import make_2lln
from je import run_JE
```

Pure-Python engine: use r ≤ 100 and short horizons interactively; for
r = 300–10⁴ use the compiled general engine `sim/colab_engine.py` (compiles
`sim/genkern.c` with Colab's gcc at import; see the cell at the end).

The code computes exact 𝕄/λ̃/b_max for every J, asserts feasibility
(no silent projection), raises when r is too small, accepts Condition 2 also in the
paper's (ᾱ, β̄) form, and uses the exact dual-vertex ĥ and the √dt-extrapolated HGI_E benchmark.
Everything is Definition 2.3 verbatim; `E_mask=0` / `vb_norm=…` is the labeled
**control experiment (outside Definition 2.3)**, never a recommended setting.

## Part 1 — the files

### `rsn.py` — the network class: arbitrary (K, α, β, h), Conditions 2 & 3, θ = Kη, Σ

In [ ]:
%%writefile rsn.py
# Standalone module with plain imports; pure Python. The compiled kernels are in sim/.

"""
rsn.py — general Resource Sharing Network (I resources, J job types).

Implements the model of Budhiraja & Johnson (2024), "Simple form control
policies for resource sharing networks with HGI performance", for an
arbitrary network (K, alpha, beta, h); the toy of Example 2.2 is `make_2lln()`.
Verifies the paper's standing assumptions programmatically:

  Condition 2 (heavy traffic / stability), in the paper's form:
      alpha^r = alpha + alpha_bar/r,   beta^r = beta + beta_bar/r,
      rho = alpha/beta,  C := K rho  (criticality is automatic by choice of C),
      eta = beta^{-2} (alpha_bar beta - alpha beta_bar)  [= lim r (rho^r - rho)],
      theta = K eta < 0  (drift of the limiting workload RBM) — asserted.
    Input is EITHER eta (default; the canonical split alpha_bar = beta*eta,
    beta_bar = 0, so rho^r = rho + eta/r exactly and beta^r = beta) OR the pair
    (alpha_bar, beta_bar). Only eta enters the limit (theta, Sigma, HGI); the
    split changes the rth system at O(1/r^2) in rho^r. The simulator must use
    beta^r (it does: `betas_r`).
    Standing assumption on the scale (text after Definition 2.3):
    |rho^r - rho| < rho*/4 componentwise — `check_r(r)` raises otherwise.

  Condition 3 (local traffic): every resource i has a job type whose
      K-column is e_i. Recorded as s_i (used by the constructive v^b
      fallback and required for the workload state space to be all of R^I_+).

Covariance of the limiting RBM (exponential primitives):
      Sigma = K M (Sigma_u + Sigma_v R) M^T K^T,
  where Sigma_u, Sigma_v are the RENEWAL VARIANCE RATES
      Sigma_u_j = alpha_j^3 (sigma_u_j)^2  = alpha_j   (exponential),
      Sigma_v_j = beta_j^3  (sigma_v_j)^2  = beta_j    (exponential),
  R = diag(rho), M = diag(1/beta). For alpha=beta=1 this is 2 K K^T.
"""

import numpy as np


class RSN:
    def __init__(self, K, alpha, beta, h, eta=None, name="RSN",
                 alpha_bar=None, beta_bar=None, scv_u=None, scv_v=None):
        self.name  = str(name)
        self.K     = np.asarray(K, dtype=float)
        self.I, self.J = self.K.shape
        self.alpha = np.asarray(alpha, dtype=float).reshape(self.J)
        self.beta  = np.asarray(beta,  dtype=float).reshape(self.J)
        self.h     = np.asarray(h,     dtype=float).reshape(self.J)

        if np.any(self.alpha <= 0) or np.any(self.beta <= 0) or np.any(self.h <= 0):
            raise ValueError("alpha, beta, h must be strictly positive")
        if not np.all(np.isin(self.K, (0.0, 1.0))):
            raise ValueError("K must be a 0/1 incidence matrix")
        if np.any(self.K.sum(axis=0) == 0):
            raise ValueError("every job type must use at least one resource")

        # ---- nominal load and capacities (criticality by construction) ----
        self.rho      = self.alpha / self.beta
        self.C        = self.K @ self.rho
        self.rho_star = float(np.min(self.rho))
        self.M        = np.diag(1.0 / self.beta)
        self.R        = np.diag(self.rho)          # diag(rho): NOT routing!

        # ---- Condition 3: local traffic ----
        self.local_types = self._find_local_types()   # s_i for each resource i
        if np.linalg.matrix_rank(self.K) != self.I:
            raise ValueError("K must have full row rank (implied by Condition 3)")

        # ---- Condition 2: eta, or the paper's (alpha_bar, beta_bar) ----
        self.eta, self.alpha_bar, self.beta_bar = self._condition2(eta, alpha_bar, beta_bar)
        self.theta = self.K @ self.eta
        if not np.all(self.theta < 0):
            raise ValueError(
                f"theta = K eta = {self.theta} is not strictly negative (Condition 2): the "
                f"limiting RBM has no stationary distribution and HGI_E is undefined. "
                f"Choose eta (equivalently alpha_bar, beta_bar) with K eta < 0.")

        # ---- RBM covariance (renewal variance rates) ----
        # Sigma_u_j = alpha_j^3 sigma_u_j^2 = alpha_j * scv_u_j,  Sigma_v_j = beta_j^3 sigma_v_j^2
        # = beta_j * scv_v_j  (scv = squared coefficient of variation; 1 for exponential,
        # 1/k for Erlang-k, > 1 for hyperexponential).
        self.scv_u = (np.ones(self.J) if scv_u is None
                      else np.broadcast_to(np.asarray(scv_u, float), (self.J,)).copy())
        self.scv_v = (np.ones(self.J) if scv_v is None
                      else np.broadcast_to(np.asarray(scv_v, float), (self.J,)).copy())
        if np.any(self.scv_u <= 0) or np.any(self.scv_v <= 0):
            raise ValueError("scv_u, scv_v must be positive")
        Sigma_u = np.diag(self.alpha * self.scv_u)
        Sigma_v = np.diag(self.beta * self.scv_v)
        self.Sigma = self.K @ self.M @ (Sigma_u + Sigma_v @ self.R) \
                     @ self.M.T @ self.K.T

    # ------------------------------------------------------------------
    def _condition2(self, eta, alpha_bar, beta_bar):
        """Resolve (eta) or (alpha_bar, beta_bar) into all three, consistently."""
        J = self.J
        if alpha_bar is None and beta_bar is None:
            eta = (np.full(J, -0.5) if eta is None
                   else np.asarray(eta, dtype=float).reshape(J))
            return eta, self.beta * eta, np.zeros(J)            # canonical split
        a_bar = (np.zeros(J) if alpha_bar is None
                 else np.asarray(alpha_bar, dtype=float).reshape(J))
        b_bar = (np.zeros(J) if beta_bar is None
                 else np.asarray(beta_bar, dtype=float).reshape(J))
        eta_from_bars = (a_bar * self.beta - self.alpha * b_bar) / self.beta ** 2
        if eta is not None:
            eta = np.asarray(eta, dtype=float).reshape(J)
            if not np.allclose(eta, eta_from_bars, rtol=1e-9, atol=1e-12):
                raise ValueError(
                    f"eta={eta} is inconsistent with (alpha_bar, beta_bar): "
                    f"beta^-2 (alpha_bar beta - alpha beta_bar) = {eta_from_bars}")
        return eta_from_bars, a_bar, b_bar

    def _find_local_types(self):
        local = {}
        for i in range(self.I):
            for j in range(self.J):
                col = self.K[:, j]
                if col[i] == 1.0 and col.sum() == 1.0:
                    local[i] = j
                    break
        missing = [i for i in range(self.I) if i not in local]
        if missing:
            raise ValueError(
                f"Local traffic condition (Condition 3) fails for resources "
                f"{missing}: each resource needs a job type using it alone.")
        return local

    # ------------------------------------------------------------------
    def alphas_r(self, r):
        """alpha^r = alpha + alpha_bar/r  (canonical split: alpha + beta*eta/r)."""
        return self.alpha + self.alpha_bar / float(r)

    def betas_r(self, r):
        """beta^r = beta + beta_bar/r  (canonical split: beta_bar = 0, beta^r = beta)."""
        return self.beta + self.beta_bar / float(r)

    def rhos_r(self, r):
        """rho^r = alpha^r / beta^r; r (rho^r - rho) -> eta."""
        return self.alphas_r(r) / self.betas_r(r)

    def check_r(self, r):
        """
        The paper's standing assumption on the scale (text after Definition 2.3):
        |rho^r - rho| < rho*/4 componentwise, rho^r = alpha^r/beta^r. Raises
        ValueError if r is too small (or if alpha^r, beta^r are not positive);
        returns rho^r otherwise.
        """
        r = float(r)
        if not r > 0:
            raise ValueError(f"r must be positive, got {r!r}")
        a_r, b_r = self.alphas_r(r), self.betas_r(r)
        if np.any(a_r <= 0) or np.any(b_r <= 0):
            raise ValueError(f"alpha^r={a_r}, beta^r={b_r} must be positive at r={r:g}")
        rho_r = a_r / b_r
        dev = np.abs(rho_r - self.rho)
        if not np.all(dev < self.rho_star / 4.0):
            j = int(np.argmax(dev))
            raise ValueError(
                f"r={r:g} too small for heavy traffic: |rho^r_{j} - rho_{j}| = {dev[j]:.4g} "
                f">= rho*/4 = {self.rho_star / 4.0:.4g}; need r > "
                f"{self.r_min_heavy_traffic():.4g} (paper: |rho^r - rho| < rho*/4).")
        return rho_r

    def r_min_heavy_traffic(self):
        """Smallest r with |rho^r - rho| < rho*/4 (exact 4 max|eta|/rho* when beta^r = beta)."""
        def ok(rr):
            a_r, b_r = self.alphas_r(rr), self.betas_r(rr)
            return bool(np.all(a_r > 0) and np.all(b_r > 0)
                        and np.all(np.abs(a_r / b_r - self.rho) < self.rho_star / 4.0))
        r = max(4.0 * float(np.max(np.abs(self.eta))) / self.rho_star, 1e-12)
        if ok(r * (1 + 1e-9)) and not ok(r * (1 - 1e-9)):
            return r                                   # the canonical-split formula is exact
        lo = hi = r                                    # otherwise bracket and bisect
        while not ok(hi):
            hi *= 2.0
        while ok(lo) and lo > 1e-12:
            lo /= 2.0
        for _ in range(80):
            mid = 0.5 * (lo + hi)
            lo, hi = (lo, mid) if ok(mid) else (mid, hi)
        return hi

    def project_capacity(self, b, max_iter=20, tol=1e-10):
        """
        Project allocation b onto {b >= 0 : K b <= C} by per-station scaling.
        DIAGNOSTIC ONLY: the policy does not call this (Definition 2.3's x(z)
        is feasible by construction and policy.allocation asserts it).
        """
        b = np.asarray(b, dtype=float).clip(min=0.0)
        for _ in range(max_iter):
            usage = self.K @ b
            over  = usage - self.C
            if np.all(over <= tol):
                break
            for i in range(self.I):
                if over[i] > tol and usage[i] > 0.0:
                    A_i = self.K[i] > 0.0
                    b[A_i] *= float(self.C[i] / usage[i])
        return b.clip(min=0.0)

    def workload(self, q):
        """W = K M q  (I-dimensional workload from queue lengths)."""
        return self.K @ self.M @ np.asarray(q, dtype=float)

    def __repr__(self):
        split = ("canonical split (alpha_bar = beta*eta, beta_bar = 0)"
                 if np.all(self.beta_bar == 0) else
                 f"alpha_bar={self.alpha_bar}  beta_bar={self.beta_bar}")
        return (f"{self.name}(I={self.I}, J={self.J})\n"
                f"  alpha={self.alpha}\n  beta ={self.beta}\n"
                f"  h    ={self.h}\n  rho  ={self.rho}  C={self.C}\n"
                f"  eta  ={self.eta}  [{split}]\n  theta={self.theta}  [stable]\n"
                f"  local types s_i = {self.local_types}")


def make_2lln(h=(1.0, 1.0, 1.0), eta=(-0.5, -0.5, -0.5), **cond2):
    """The paper's Example 2.2 toy: S1 serves B1,B3; S2 serves B2,B3.
    `cond2` may carry alpha_bar/beta_bar instead of eta (then pass eta=None)."""
    K = [[1.0, 0.0, 1.0],
         [0.0, 1.0, 1.0]]
    return RSN(K, alpha=[1, 1, 1], beta=[1, 1, 1], h=h, eta=eta, name="2LLN", **cond2)


def make_3res_5type(h=(1.0, 1.0, 1.0, 1.0, 1.0), eta=None, **cond2):
    """
    A larger test network: I=3 resources, J=5 job types.
      B1, B2, B3 local to S1, S2, S3;  B4 uses S1+S2;  B5 uses S2+S3.
    dim ker(K) = 2, so C^h_K is 1-dimensional and the cost geometry is
    nontrivial for generic h.
    """
    K = [[1.0, 0.0, 0.0, 1.0, 0.0],
         [0.0, 1.0, 0.0, 1.0, 1.0],
         [0.0, 0.0, 1.0, 0.0, 1.0]]
    if eta is None and not cond2:
        eta = [-0.5] * 5
    return RSN(K, alpha=[1] * 5, beta=[1] * 5, h=h, eta=eta, name="3res5type", **cond2)

### `offline.py` — the offline pass: u_{J−I}, λ, the set 𝕄, v^c(z), v^b(z) by LP, feasibility assertions

In [ ]:
%%writefile offline.py
# Standalone module with plain imports; pure Python. The compiled kernels are in sim/.

"""
offline.py — the offline pass: the vectors of Definition 2.3 for an arbitrary network.

Everything the paper leaves as "there exists a vector such that ..." is
computed here, once, before simulation:

  1. cost geometry: orthonormal kernel basis N of ker(K) via SVD,
     g = h*beta, g_ker = N N^T g,
       Case A (trivial):   g_ker = 0  ->  v^c = 0, M_set = {} (only v^b active)
       Case B (nontrivial): u = -g_ker/|g_ker|, lambda = g.u = -|g_ker| < 0
  2. the set M of Z-configurations where cost reduction is possible:
       z in M  <=>  max{ v.u : Kv=0, v_j>=0 if z_j=1, |v|<=1 } > tol   (LP-c)
  3. cost-reduction vectors v^c(z): scaled LP-c maximizer, obeying the
     size condition rho - v^c > rho*/2 componentwise;
     lambda_c(z) = lambda*(v^c(z).u) < 0,  lambda_tilde = max_z lambda_c(z)
  4. boundary-repulsion vectors v^b(z): LP-b
       max t  s.t. (Kv)_i = 0 for i in A_z, v_j >= t if z_j = 1,
                   |v| <= 1, t >= 0,
     scaled to norm gamma*b_max with
       b_max = min(rho*/4, |lambda_tilde| / (4 |beta|_2 |h|_2))   (Case B)
       b_max = rho*/4                                             (Case A)
     with the paper's constructive fallback (proof of Prop 2.3) if the LP
     ever returns t* ~ 0.
  5. feasibility assertions for every z:
       x(z) = rho - v^c(z) 1{z in M} - v^b(z)  satisfies
       K x <= C  and  x >= rho*/4 > 0.

Pattern-dependent cap (LABELED EXPERIMENT, off by default).
`vb_cap="per_pattern"` replaces the uniform eq. (6) cap by
    ||v^b(z)|| <= min(rho*/4, |lambda_c(z)| / (4 |beta|_2 |h|_2))   for z in M,
    ||v^b(z)|| <= rho*/4                                             for z not in M.
Rationale: in the proof of Proposition 3.5 (paper section 7.2) the size of
v^b enters only through step (47), which needs, for Z^r(s) in M,
    h beta . (v^c(z) + v^b(z)) <= -|lambda~| + |h||beta| max_z|v^b(z)| <= -3/4 |lambda~|;
with the per-pattern cap one gets the stronger pointwise chain
    h beta . (v^c(z) + v^b(z)) <= -3/4 |lambda_c(z)| <= -3/4 |lambda~|
(see `check_step47`). Off M, (47) is never invoked (Lemma 7.4 / Prop 7.7
handle those configurations) and only x >= rho*/4 (Prop 2.6(c)) is needed.
Everything produced with this option is OUTSIDE eq. (6) of Definition 2.3
and is labeled so (`label`, `summary()`); `b_max` always reports the eq. (6)
value, the per-pattern value is `b_cap(z)`.

Enumeration. M is a down-set of {0,1}^J: if z in M then every z' <= z
(componentwise) is in M, because the LP-c feasible region only grows when
constraints v_j >= 0 are dropped. Equivalently, if z is NOT in M then no
z' >= z is. The pass therefore walks the patterns by increasing popcount and
solves LP-c only for the patterns all of whose immediate predecessors
(z minus one bit) are members. The LP-c solves are exactly the ones the
brute-force enumeration would have found nonzero, plus the minimal
non-members, so M, every v^c(z), lambda_tilde and hence b_max are identical
to the full 2^J enumeration — for every J, with no "lazy" approximation.

The v^b table (one LP-b per pattern, 2^J of them) is precomputed and
asserted in full when J <= max_enumerate (default 16). Above that it is
filled on first visit and EVERY entry is asserted (K x <= C, x >= rho*/4)
the moment it is created; b_max is already exact at that point. Call
`enumerate_vb()` to force the full table.
"""

import itertools
import numpy as np
from scipy.optimize import linprog

_TOL = 1e-9


def kernel_basis(K, tol=1e-10):
    """Orthonormal basis (columns) of ker(K) via SVD."""
    K = np.asarray(K, dtype=float)
    _, s, Vt = np.linalg.svd(K)
    rank = int(np.sum(s > tol * max(K.shape) * (s[0] if len(s) else 1.0)))
    return Vt[rank:].T          # J x (J - rank)


class OfflineTables:
    VB_CAPS = ("uniform", "per_pattern")

    def __init__(self, rsn, gamma=0.9, max_enumerate=16, tol=_TOL,
                 vb_norm=None, gamma_c=None, gamma_b=None, vb_cap="uniform"):
        """
        gamma_c: safety factor for the v^c scaling toward the size condition
                 rho - v^c > rho*/2. NOTE (within Definition 2.3): b_max =
                 min(rho*/4, |lambda~|/(4|beta||h|)) depends on the CHOSEN
                 v^c through lambda~, so taking gamma_c close to 1 legally
                 enlarges the admissible ||v^b||. Default: `gamma`.
        gamma_b: fraction of b_max used for ||v^b(z)||. Default: `gamma`.
        vb_norm: CONTROL-EXPERIMENT override of ||v^b(z)|| ignoring b_max.
        Values above b_max leave the paper's construction (eq. (6)) — a run
        using it is a labeled control experiment OUTSIDE Definition 2.3, never
        a default; feasibility (Kx <= C, x >= rho*/4) is still asserted.
        max_enumerate: J above which the v^b table is filled on demand
        instead of precomputed. M, lambda~ and b_max are always exact.
        vb_cap: "uniform" (default) = eq. (6), ||v^b(z)|| = gamma_b * b_max for
        every z; "per_pattern" = the LABELED EXPERIMENT, OUTSIDE
        eq. (6): ||v^b(z)|| = gamma_b * b_cap(z) with
        b_cap(z) = min(rho*/4, |lambda_c(z)|/(4|beta||h|)) on M, rho*/4 off M.
        It preserves the inequality that step (47) of the proof needs
        (`check_step47`); whether it is admissible for Theorems 2.4-2.5 is
        the question put to the advisor (docs/note_for_advisor.md).
        """
        self.rsn     = rsn
        self.gamma   = float(gamma)
        self.gamma_c = float(gamma if gamma_c is None else gamma_c)
        self.gamma_b = float(gamma if gamma_b is None else gamma_b)
        self.tol     = float(tol)
        self.vb_norm = None if vb_norm is None else float(vb_norm)
        if vb_cap not in self.VB_CAPS:
            raise ValueError(f"vb_cap must be one of {self.VB_CAPS}, got {vb_cap!r}")
        self.vb_cap = vb_cap
        self.max_enumerate = int(max_enumerate)

        K, beta, h, rho = rsn.K, rsn.beta, rsn.h, rsn.rho
        self.N = kernel_basis(K)                       # J x (J-I)
        g = h * beta
        g_ker = self.N @ (self.N.T @ g)
        norm = float(np.linalg.norm(g_ker))

        self.trivial = norm < 1e-10                    # Case A
        if self.trivial:
            self.u, self.lam = None, 0.0
        else:                                          # Case B
            self.u   = -g_ker / norm                   # steepest cost descent in ker(K)
            self.lam = float(g @ self.u)               # = -norm < 0

        self._M         = set()     # members of M (tuples)
        self._vc_cache  = {}        # z -> v^c(z), members only
        self._lam_c     = {}        # z -> lambda_c(z), members only
        self._inM_cache = {}        # z -> bool, every z on which LP-c was solved
        self._vb_cache  = {}        # z -> v^b(z)
        self.lam_tilde  = None
        self.b_max      = None
        self.vb_enumerated = False

        self._build_M()                                # exact M, lambda~ (pruned)
        self._get_b_max()                              # exact b_max, eq. (6)
        if rsn.J <= self.max_enumerate:
            self.enumerate_vb()                        # full table, all asserted

    # ------------------------------------------------------------------
    # LP-c: cost-reduction direction for configuration z
    # ------------------------------------------------------------------
    def _solve_lp_c(self, z):
        """max v.u  s.t. Kv=0, v_j >= 0 where z_j=1, -1<=v<=1. Returns (t*, v*)."""
        if self.trivial:
            return 0.0, None
        rsn = self.rsn
        bounds = [(0.0, 1.0) if z[j] else (-1.0, 1.0) for j in range(rsn.J)]
        res = linprog(c=-self.u, A_eq=rsn.K, b_eq=np.zeros(rsn.I),
                      bounds=bounds, method="highs")
        if not res.success:
            return 0.0, None
        return float(-res.fun), np.asarray(res.x)

    def _make_vc(self, z):
        """v^c(z): scaled LP-c maximizer meeting rho - v^c > rho*/2."""
        t_star, v_star = self._solve_lp_c(z)
        in_M = (t_star > self.tol)
        if not in_M:
            return False, np.zeros(self.rsn.J)
        rho, rho_star = self.rsn.rho, self.rsn.rho_star
        pos = v_star > self.tol
        if np.any(pos):
            s = self.gamma_c * float(np.min((rho[pos] - rho_star / 2.0) / v_star[pos]))
        else:
            s = 1.0
        return True, s * v_star

    # ------------------------------------------------------------------
    # The set M by enumeration with monotone pruning (M is a down-set)
    # ------------------------------------------------------------------
    def _solve_member(self, zint):
        """Solve LP-c for the pattern zint (bit j = z_j); record the result."""
        J = self.rsn.J
        z = tuple((zint >> j) & 1 for j in range(J))
        in_M, vc = self._make_vc(z)
        self._inM_cache[z] = in_M
        if in_M:
            self._M.add(z)
            self._vc_cache[z] = vc
            self._lam_c[z] = self.lam * float(vc @ self.u)
        return in_M

    def _build_M(self):
        """
        Walk {0,1}^J by popcount. A pattern is a candidate (gets an LP-c
        solve) only if all its immediate predecessors are members; by the
        down-set property every member is reached this way and every
        pattern that is skipped is a non-member. Exact for every J.
        """
        if self.trivial:
            self.lam_tilde = None
            return
        J = self.rsn.J
        level = [0] if self._solve_member(0) else []
        for _k in range(1, J + 1):
            if not level:
                break
            prev = set(level)
            cands = set()
            for m in level:
                for j in range(J):
                    bit = 1 << j
                    if m & bit:
                        continue
                    c = m | bit
                    if c in cands:
                        continue
                    ok, cc = True, c
                    while cc:                      # every c minus one bit in prev?
                        b = cc & -cc
                        if (c ^ b) not in prev:
                            ok = False
                            break
                        cc ^= b
                    if ok:
                        cands.add(c)
            level = [c for c in sorted(cands) if self._solve_member(c)]
        self.lam_tilde = max(self._lam_c.values()) if self._lam_c else None

    # ------------------------------------------------------------------
    # LP-b: boundary repulsion for configuration z
    # ------------------------------------------------------------------
    def _A_z(self, z):
        """Busy resources: i such that some j with K_ij=1 has z_j=0."""
        K = self.rsn.K
        return [i for i in range(self.rsn.I)
                if any(K[i, j] == 1.0 and not z[j] for j in range(self.rsn.J))]

    def _solve_lp_b(self, z):
        """max t s.t. (Kv)_i=0 (i in A_z), v_j>=t (z_j=1), |v|<=1, 0<=t<=1."""
        rsn = self.rsn
        J = rsn.J
        A_z = self._A_z(z)
        ones_idx = [j for j in range(J) if z[j]]
        # variables: [v_1..v_J, t]
        c = np.zeros(J + 1); c[-1] = -1.0
        A_eq = None; b_eq = None
        if A_z:
            A_eq = np.hstack([rsn.K[A_z, :], np.zeros((len(A_z), 1))])
            b_eq = np.zeros(len(A_z))
        A_ub, b_ub = None, None
        if ones_idx:
            rows = []
            for j in ones_idx:                 # t - v_j <= 0
                row = np.zeros(J + 1); row[j] = -1.0; row[-1] = 1.0
                rows.append(row)
            A_ub = np.vstack(rows); b_ub = np.zeros(len(rows))
        bounds = [(-1.0, 1.0)] * J + [(0.0, 1.0)]
        res = linprog(c=c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                      bounds=bounds, method="highs")
        if not res.success:
            return 0.0, None
        return float(res.x[-1]), np.asarray(res.x[:J])

    def _vb_constructive(self, z):
        """Paper's constructive proof of Prop 2.3 (fallback if LP-b degenerate)."""
        rsn = self.rsn
        J = rsn.J
        A_z = self._A_z(z)
        S = {rsn.local_types[i] for i in A_z}
        v = np.zeros(J)
        for j in range(J):
            if j in S:
                continue
            v[j] = 1.0 if z[j] else -float(J)
        for i in A_z:
            s_i = rsn.local_types[i]
            v[s_i] = -float(sum(rsn.K[i, j] * v[j] for j in range(J) if j != s_i))
        for i in range(rsn.I):
            s_i = rsn.local_types[i]
            if i not in A_z and z[s_i]:
                v[s_i] = 1.0
        return v

    def _make_vb(self, z):
        if not any(z):                          # z = 0: no repulsion needed
            return np.zeros(self.rsn.J)
        t_star, v_star = self._solve_lp_b(z)
        if v_star is None or t_star <= self.tol:
            v_star = self._vb_constructive(z)
        target = (self.vb_norm if self.vb_norm is not None
                  else self.gamma_b * self.b_cap(z))
        return target * v_star / float(np.linalg.norm(v_star))

    # ------------------------------------------------------------------
    def _get_b_max(self):
        """b_max = min(rho*/4, |lambda~|/(4|beta|_2|h|_2))  — eq. (6)."""
        if self.b_max is not None:
            return self.b_max
        rsn = self.rsn
        cap = rsn.rho_star / 4.0
        if not self.trivial and self.lam_tilde is not None:
            cap = min(cap, abs(self.lam_tilde)
                      / (4.0 * np.linalg.norm(rsn.beta) * np.linalg.norm(rsn.h)))
        self.b_max = cap
        return cap

    def b_cap(self, z):
        """Admissible ||v^b(z)|| before the gamma_b factor.

        "uniform":     b_max = min(rho*/4, |lambda~|/(4|beta||h|))        (eq. (6))
        "per_pattern": min(rho*/4, |lambda_c(z)|/(4|beta||h|)) for z in M,
                       rho*/4 for z not in M                     (OUTSIDE eq. (6))
        The two agree at every z in M attaining lambda~ and whenever rho*/4 binds.
        """
        if self.vb_cap == "uniform":
            return self._get_b_max()
        z = tuple(int(v) for v in z)
        rsn = self.rsn
        cap = rsn.rho_star / 4.0
        if z in self._M:
            cap = min(cap, abs(self._lam_c[z])
                      / (4.0 * np.linalg.norm(rsn.beta) * np.linalg.norm(rsn.h)))
        return cap

    @property
    def label(self):
        """How to label every output derived from these tables."""
        if self.vb_norm is not None:
            return f"CONTROL EXPERIMENT outside Definition 2.3: vb_norm={self.vb_norm}"
        if self.vb_cap == "per_pattern":
            return "outside eq. (6): per-pattern cap |lambda_c(z)|/(4|beta||h|) on M, rho*/4 off M"
        return "Definition 2.3, eq. (6)"

    def check_step47(self, tol=1e-9):
        """
        The inequality that step (47) of the proof of Proposition 3.5 needs, for
        every z in M (it is only applied there):
            h beta . (v^c(z) + v^b(z)) <= -3/4 |lambda_c(z)| <= -3/4 |lambda~|.
        Returns a dict with the worst (largest) value of the left side over M, the
        per-pattern margins, and `ok`. Raises AssertionError if it fails.
        """
        if self.trivial or not self._M:
            return dict(ok=True, n=0, worst_lhs=None, min_margin=None, note="Case A: M empty")
        g = self.rsn.h * self.rsn.beta
        lam_t = abs(self.lam_tilde)
        worst, margins, margins_tilde = -np.inf, [], []
        for z in self._M:
            lhs = float(g @ (self._vc_cache[z] + self.vb(z)))
            lam_z = abs(self._lam_c[z])
            margins.append(-0.75 * lam_z - lhs)           # >= 0 required
            margins_tilde.append(-0.75 * lam_t - lhs)     # >= 0 required (weaker)
            worst = max(worst, lhs)
        out = dict(ok=bool(min(margins) >= -tol and min(margins_tilde) >= -tol), n=len(self._M),
                   worst_lhs=worst, min_margin=float(min(margins)),
                   min_margin_tilde=float(min(margins_tilde)), lam_tilde=self.lam_tilde)
        if not out["ok"]:
            raise AssertionError(f"step-(47) inequality violated: {out}")
        return out

    def push_stats(self):
        """||v^b(z)|| over the tabulated patterns (z != 0): overall, on M and off M.
        Call enumerate_vb() first for the full table."""
        norms = {z: float(np.linalg.norm(v)) for z, v in self._vb_cache.items() if any(z)}
        def stats(vals):
            vals = np.array(sorted(vals))
            return (dict(n=len(vals), min=float(vals.min()), median=float(np.median(vals)),
                         max=float(vals.max())) if len(vals) else dict(n=0))
        return dict(all=stats(norms.values()),
                    in_M=stats([v for z, v in norms.items() if z in self._M]),
                    off_M=stats([v for z, v in norms.items() if z not in self._M]),
                    b_max_eq6=self._get_b_max(), vb_cap=self.vb_cap, label=self.label)

    def enumerate_vb(self):
        """Precompute v^b(z) for all 2^J patterns and assert every x(z)."""
        for z in itertools.product((0, 1), repeat=self.rsn.J):
            if z not in self._vb_cache:
                self._vb_cache[z] = self._make_vb(z)
        self.vb_enumerated = True
        self.assert_feasibility()

    # ------------------------------------------------------------------
    # public lookups
    # ------------------------------------------------------------------
    def in_M(self, z):
        return tuple(int(v) for v in z) in self._M

    def vc(self, z):
        z = tuple(int(v) for v in z)
        return self._vc_cache.get(z, np.zeros(self.rsn.J))

    def lam_c(self, z):
        """lambda_c(z) = lambda (v^c(z).u) for z in M, else 0."""
        return self._lam_c.get(tuple(int(v) for v in z), 0.0)

    def vb(self, z):
        z = tuple(int(v) for v in z)
        if z not in self._vb_cache:
            self._vb_cache[z] = self._make_vb(z)
            self._check_x(z)                    # every lazily created entry
        return self._vb_cache[z]

    def x(self, z):
        """Nominal service-rate vector x(z) = rho - v^c 1{z in M} - v^b(z)."""
        z = tuple(int(v) for v in z)
        x = self.rsn.rho.copy()
        if z in self._M:
            x = x - self._vc_cache[z]
        return x - self.vb(z)

    @property
    def M_set(self):
        return sorted(self._M)

    @property
    def n_lp_c(self):
        """Number of LP-c solves the pruned enumeration needed (|M| + minimal non-members)."""
        return len(self._inM_cache)

    # ------------------------------------------------------------------
    def _check_x(self, z, tol=1e-8):
        x = self.x(z)
        Kx = self.rsn.K @ x
        if not np.all(Kx <= self.rsn.C + tol):
            raise AssertionError(
                f"K x(z) <= C violated at z={z}: Kx={Kx}, C={self.rsn.C}")
        if not np.all(x >= self.rsn.rho_star / 4.0 - tol):
            raise AssertionError(f"x(z) >= rho*/4 violated at z={z}: x={x}")

    def assert_feasibility(self, tol=1e-8):
        """Paper Prop 2.6-style checks: Kx <= C and x >= rho*/4 for every tabulated z."""
        for z in list(self._vb_cache):
            self._check_x(z, tol)

    def summary(self):
        lines = [f"Offline pass for {self.rsn.name} "
                 f"(I={self.rsn.I}, J={self.rsn.J}, dim ker K={self.N.shape[1]})"]
        if self.trivial:
            lines.append("  Case A (trivial): h*beta ⟂ ker(K); v^c ≡ 0, M = {}")
        else:
            lines.append(f"  u_(J-I)   = {np.round(self.u, 6)}")
            lines.append(f"  lambda    = {self.lam:.6f}")
            if len(self._M) <= 32:
                lines.append(f"  M set     = {self.M_set}")
            else:
                lines.append(f"  |M|       = {len(self._M)} of {2 ** self.rsn.J}")
            lines.append(f"  LP-c solves = {self.n_lp_c} (pruned enumeration)")
            lines.append(f"  lambda~   = {self.lam_tilde:.6f}" if self.lam_tilde
                         is not None else "  lambda~   = (M empty)")
        lines.append(f"  b_max     = {self._get_b_max():.6f}  (eq. (6); gamma_c={self.gamma_c}, gamma_b={self.gamma_b})")
        if self.vb_cap == "per_pattern":
            caps = [self.b_cap(z) for z in self._M] if self._M else []
            lines.append("  *** vb_cap = per_pattern: OUTSIDE eq. (6) — labeled experiment ***")
            lines.append(f"  b_cap(z)  = |lambda_c(z)|/(4|beta||h|) on M" +
                         (f": {min(caps):.6f} .. {max(caps):.6f}" if caps else "") +
                         f"; rho*/4 = {self.rsn.rho_star / 4:.6f} off M")
        if self.vb_norm is not None:
            lines.append(f"  *** vb_norm = {self.vb_norm}: CONTROL EXPERIMENT outside Definition 2.3 ***")
        lines.append("  v^b table = " + ("all %d patterns precomputed and asserted" % 2 ** self.rsn.J
                                         if self.vb_enumerated else
                                         "filled on demand, each entry asserted on creation"))
        return "\n".join(lines)

### `je_stats.py` — batch-means confidence intervals

In [ ]:
%%writefile je_stats.py
# Standalone module with plain imports; pure Python. The compiled kernels are in sim/.

"""
je_stats.py — statistical utilities for ergodic simulation output.

batch_means_ci: nonoverlapping batch-means confidence interval for the mean
of a stationary, autocorrelated series (the standard method for steady-state
simulation output analysis). The series here is the evenly-time-spaced
snapshot read-out of an ergodic run; batches are valid when the batch span
well exceeds the process mixing time (for the workload RBM, mixing ~
Sigma_ii/theta_i^2 diffusion units — with default settings each batch spans
(1-burn)*T/n_batches diffusion units; check batch_span_ok in the output).
"""

import numpy as np
from scipy import stats as _st


def batch_means_ci(series, n_batches=20, conf=0.95):
    """
    Returns dict(mean, half_width, se, n_batches). The half-width uses the
    Student-t quantile on n_batches-1 degrees of freedom applied to the
    batch means — the standard steady-state output-analysis estimator.
    """
    x = np.asarray(series, dtype=float).ravel()
    n = x.size
    nb = int(n_batches)
    if n < 2 * nb:
        raise ValueError(f"need >= {2*nb} points for {nb} batches, got {n}")
    m = n // nb
    b = x[: m * nb].reshape(nb, m).mean(axis=1)
    mean = float(b.mean())
    se = float(b.std(ddof=1) / np.sqrt(nb))
    t = float(_st.t.ppf(0.5 + conf / 2.0, nb - 1))
    return dict(mean=mean, half_width=t * se, se=se, n_batches=nb)

### `policy.py` — Definition 2.3 online (table-driven)

In [ ]:
%%writefile policy.py
# Standalone module with plain imports; pure Python. The compiled kernels are in sim/.

"""
policy.py — the control policy of Definition 2.3, for an arbitrary RSN.

Online state per job type j: queue length Q_j and a hysteresis flag
E_j in {0,1}:
    E_j: 0 -> 1  when Q_j drops below  L = c1~ r^kappa   (service stops),
    E_j: 1 -> 0  when Q_j climbs to    U = c2~ r^kappa   (service resumes),
with c_i~ = (min_j beta_j) * c_i. At every instant:

    Z_j = 1{ Q_j < U }
    x   = rho - v^c(Z) 1{Z in M} - v^b(Z)      (offline lookup tables)
    b_j = x_j * 1{E_j = 0}

The whole online policy is one table lookup and a multiplication; all the
geometry (u, M, v^c, v^b) lives in offline.OfflineTables.

Checks (the paper's standing assumptions, raised — never warned or patched):
  * x(Z) must satisfy K x <= C and x >= 0 — true by construction (asserted
    offline for every tabulated pattern), so `allocation` ASSERTS it instead of
    projecting b back onto the capacity set. A violation raises FeasibilityError and is counted in
    `n_violations`.
  * r must be large enough: |rho^r - rho| < rho*/4 componentwise (RSN.check_r)
    and (c2~ - c1~) r^kappa > 1 (a hysteresis band of at least one job).
    Both raise ValueError.
"""

import numpy as np
from offline import OfflineTables


class FeasibilityError(AssertionError):
    """x(Z) left the capacity set — impossible by construction; a bug if seen."""


class GeneralThresholdPolicy:
    def __init__(self, rsn, c1=1.0, c2=4.0, kappa=0.15, gamma=0.9,
                 tables=None, E_mask=None):
        assert 0.0 < c1 < c2, "need 0 < c1 < c2 (hysteresis)"
        assert 0.0 < kappa < 0.25, "paper requires kappa in (0, 1/4)"
        self.rsn = rsn
        self.c1, self.c2, self.kappa = float(c1), float(c2), float(kappa)
        self.beta_min = float(np.min(rsn.beta))
        self.tables = OfflineTables(rsn, gamma=gamma) if tables is None else tables
        self.E = np.zeros(rsn.J)
        # E_mask: bitmask of buffers that use hysteresis flags (default: all).
        # E_mask=0 disables the shutoff entirely: a labeled CONTROL EXPERIMENT
        # outside Definition 2.3, never a default (see docs/findings_toy_convergence.md).
        self.E_mask = (1 << rsn.J) - 1 if E_mask is None else int(E_mask)
        self._masked = np.array([(self.E_mask >> j) & 1 for j in range(rsn.J)], dtype=bool)
        self._checked_r = None          # r for which the standing assumptions were verified
        self.n_allocations = 0
        self.n_violations = 0
        self.feas_tol = 1e-9

    # ------------------------------------------------------------------
    def check_r(self, r):
        """Raise unless r satisfies the paper's standing assumptions for this policy."""
        r = float(r)
        self.rsn.check_r(r)                                   # |rho^r - rho| < rho*/4
        band = (self.c2 - self.c1) * self.beta_min * r ** self.kappa
        if not band > 1.0:
            raise ValueError(
                f"(c2~ - c1~) r^kappa = {band:.3f} <= 1 at r={r:g}: the hysteresis band "
                f"must exceed one job (paper, after Definition 2.3). Increase r, c2 - c1 "
                f"or kappa; r > {(1.0 / ((self.c2 - self.c1) * self.beta_min)) ** (1.0 / self.kappa):.3g} is needed.")
        self._checked_r = r

    def thresholds(self, r):
        if r != self._checked_r:
            self.check_r(r)
        rk = float(r) ** self.kappa
        return self.c1 * self.beta_min * rk, self.c2 * self.beta_min * rk

    def reset_E(self, value=0.0):
        self.E[:] = float(value)

    # ------------------------------------------------------------------
    def allocation(self, q, r):
        """Returns (b, Z, E). b >= 0 and K b <= C hold by construction and are asserted."""
        q = np.asarray(q, dtype=float)
        L, U = self.thresholds(r)

        Z = (q < U - 1e-12).astype(int)
        masked = self._masked
        self.E[masked & (q < L - 1e-12)] = 1.0
        self.E[masked & (q >= U - 1e-12)] = 0.0
        E = self.E.copy()

        x = self.tables.x(Z)                    # rho - v^c 1{Z in M} - v^b(Z)
        self.n_allocations += 1
        rsn = self.rsn
        if np.any(x < -self.feas_tol) or np.any(rsn.K @ x > rsn.C + self.feas_tol):
            self.n_violations += 1
            raise FeasibilityError(
                f"x(Z) infeasible at Z={tuple(Z)}: x={x}, Kx={rsn.K @ x}, C={rsn.C} "
                f"(offline tables are asserted; this should be impossible)")
        b = (1.0 - E) * x
        return b, Z, E

### `simulation.py` — exact Gillespie simulation, LP-ĥ, RBM utilities + the exact-reflection benchmark

In [ ]:
%%writefile simulation.py
# Standalone module with plain imports; pure Python. The compiled kernels are in sim/.

"""
simulation.py — exact CTMC simulation, HGI benchmark, and estimators.

Scaling conventions:
  * the rth system runs with O(1) rates alpha^r = alpha + beta*eta/r —
    rates are NEVER multiplied by r;
  * the diffusion clock is t -> r^2 t: a horizon of T diffusion units is
    r^2*T physical time units;
  * read-outs are Q/r (queues) and W = K M Q / r (workloads).

The model under the policy is a CTMC (between events the allocation is
constant), so Gillespie simulation is exact — no Euler/tau-leap error.

h_hat(w) = min{ h.q : q >= 0, K M q = w } is evaluated EXACTLY and vectorized
through the dual: h_hat(w) = max_k V_k . w over the vertices V of the dual
polyhedron {y : (KM)^T y <= h} (the primal is feasible for every w >= 0 by
Condition 3, so strong duality applies and the maximum is attained at a
vertex). The vertices are enumerated once per network (C(J, I) candidate
bases); when that count exceeds `MAX_VERTEX_COMBINATIONS` the code falls back
to one LP per snapshot (never per event). Toy: V = {(1,0), (0,1)}, so
h_hat(w) = max(w_1, w_2); Example 2.1 (h = (1,1,2)): V = {(1,1)}, h_hat = w_1 + w_2.
"""

import math
import itertools
import numpy as np
from scipy.optimize import linprog

MAX_VERTEX_COMBINATIONS = 200_000


# ---------------------------------------------------------------------------
# HGI ideal cost  h_hat(w)
# ---------------------------------------------------------------------------
def hhat_lp(w, rsn):
    """LP: min h.q s.t. K M q = w, q >= 0 (reference implementation, one LP)."""
    res = linprog(c=rsn.h, A_eq=rsn.K @ rsn.M, b_eq=np.asarray(w, float),
                  bounds=[(0, None)] * rsn.J, method="highs")
    if res.success:
        return float(res.fun)
    # infeasible only from numerical round-off of w; clip and retry once
    res = linprog(c=rsn.h, A_eq=rsn.K @ rsn.M,
                  b_eq=np.maximum(np.asarray(w, float), 0.0),
                  bounds=[(0, None)] * rsn.J, method="highs")
    return float(res.fun) if res.success else float("nan")


def hhat_batch_lp(W, rsn):
    """LP h_hat for each row of W (n x I)."""
    return np.array([hhat_lp(w, rsn) for w in np.atleast_2d(W)])


def dual_vertices(rsn, max_combinations=MAX_VERTEX_COMBINATIONS):
    """
    Vertices of the dual polyhedron {y in R^I : (KM)^T y <= h}, as an (n_v x I)
    array, or None when C(J, I) > max_combinations. For w >= 0,
    h_hat(w) = max_k V_k . w  (exact).
    """
    if math.comb(rsn.J, rsn.I) > max_combinations:
        return None
    A = (rsn.K @ rsn.M).T                    # J x I: dual constraints A y <= h
    V = []
    for rows in itertools.combinations(range(rsn.J), rsn.I):
        B = A[list(rows)]
        if abs(np.linalg.det(B)) < 1e-12:
            continue
        y = np.linalg.solve(B, rsn.h[list(rows)])
        if np.all(A @ y <= rsn.h + 1e-10):
            V.append(y)
    return np.unique(np.round(np.array(V), 12), axis=0)


def make_hhat(rsn, max_combinations=MAX_VERTEX_COMBINATIONS):
    """
    Vectorized ideal cost W (n x I) -> (n,). Exact dual-vertex formula when the
    vertex enumeration is affordable, else the LP fallback. The result is cached
    on the network object (`rsn._hhat_fn`); `fn.method` is "vertices" or "lp".
    """
    fn = getattr(rsn, "_hhat_fn", None)
    if fn is not None:
        return fn
    V = dual_vertices(rsn, max_combinations)
    if V is None:
        def fn(W):
            return hhat_batch_lp(W, rsn)
        fn.method, fn.vertices = "lp", None
    else:
        VT = np.ascontiguousarray(V.T)

        def fn(W):
            return (np.atleast_2d(np.asarray(W, float)) @ VT).max(axis=1)
        fn.method, fn.vertices = "vertices", V
    rsn._hhat_fn = fn
    return fn


def hhat(w, rsn):
    """h_hat(w) for one workload vector (exact; dual vertices, LP fallback)."""
    return float(make_hhat(rsn)(np.asarray(w, float).reshape(1, -1))[0])


def hhat_batch(W, rsn):
    """h_hat for each row of W (n x I) — exact dual-vertex formula by default."""
    return np.asarray(make_hhat(rsn)(W), float)


# ---------------------------------------------------------------------------
# Exact Gillespie simulation with time-weighted ergodic accumulators
# ---------------------------------------------------------------------------
def gillespie_ergodic(rsn, pol, r, T_diff=20.0, burn_frac=0.3,
                      n_snapshots=2000, q0=None, rng=None):
    """
    Simulate the rth CTMC under `pol` to physical time r^2*T_diff.

    Returns dict with:
      JE        : time average of h.Q/r over the post-burn-in window (exact,
                  time-weighted — zero sampling variance given the path)
      HGI_path  : mean of h_hat(K M Q/r) over n_snapshots uniform times
                  (same-path ideal cost; JE - HGI_path is the SSC gap)
      ssc_gap   : JE_snap - HGI_path computed on the SAME snapshots (paired)
      E_frac    : fraction of (post-burn-in) time some E_j = 1  (idling diag.)
      util      : time-average of K b / C (per-resource utilization diag.)
      snapshots : (n_snapshots x J) queue array, for any further analysis
    """
    rng = np.random.default_rng() if rng is None else rng
    rsn.check_r(r)                                   # |rho^r - rho| < rho*/4, raises
    if hasattr(pol, "check_r"):
        pol.check_r(r)                               # (c2~ - c1~) r^kappa > 1, raises
    alpha_r, beta_r = rsn.alphas_r(r), rsn.betas_r(r)
    J = rsn.J
    T_end  = (r ** 2) * float(T_diff)
    T_burn = float(burn_frac) * T_end
    snap_times = np.linspace(T_burn, T_end, n_snapshots, endpoint=False)

    q = np.zeros(J) if q0 is None else np.asarray(q0, float).copy()
    if hasattr(pol, "reset_E"):
        pol.reset_E(0.0)

    t = 0.0
    int_cost = 0.0
    int_Eon  = 0.0
    int_util = np.zeros(rsn.I)
    snaps = np.empty((n_snapshots, J))
    next_snap = 0
    h = rsn.h

    while t < T_end:
        b, Z, E = pol.allocation(q, r)
        dep = beta_r * b * (q > 0)
        rates = np.concatenate([alpha_r, dep])
        total = rates.sum()
        t_next = t + rng.exponential(1.0 / total)

        # snapshots falling in the holding interval [t, t_next)
        while next_snap < n_snapshots and snap_times[next_snap] < t_next:
            snaps[next_snap] = q
            next_snap += 1

        # time-weighted accumulators over the post-burn-in overlap
        overlap = min(t_next, T_end) - max(t, T_burn)
        if overlap > 0:
            int_cost += overlap * float(h @ q)
            int_util += overlap * (rsn.K @ b)
            if np.any(E > 0):
                int_Eon += overlap

        t = t_next
        if t >= T_end:
            break
        # inverse-CDF event draw (faster than rng.choice with p=)
        k = int(np.searchsorted(np.cumsum(rates), rng.random() * total))
        if k < J:
            q[k] += 1.0
        else:
            k -= J
            if q[k] > 0:
                q[k] -= 1.0

    while next_snap < n_snapshots:          # horizon edge
        snaps[next_snap] = q
        next_snap += 1

    window = T_end - T_burn
    W_hat = (snaps @ (rsn.K @ rsn.M).T) / r
    hgi_path = hhat_batch(W_hat, rsn)
    je_snap  = (snaps @ h) / r
    return dict(
        JE=int_cost / window / r,
        HGI_path=float(hgi_path.mean()),
        ssc_gap=float((je_snap - hgi_path).mean()),
        E_frac=int_Eon / window,
        util=int_util / window / rsn.C,
        snapshots=snaps,
        n_violations=getattr(pol, "n_violations", 0),
    )


# ---------------------------------------------------------------------------
# RBM benchmark (limit process)
# ---------------------------------------------------------------------------
def rbm_ergodic_hgi(rsn, T=3000.0, dt=0.01, burn_frac=0.1, thin=None, rng=None):
    """
    Long-run average of h_hat(W) for dW = theta dt + Sigma^{1/2} dB reflected
    componentwise in R^I_+ by REFLECTED EULER (clip at 0 each step). Kept for
    reference only. On a grid it is the same scheme as the Skorokhod map of the
    discretized path, and it is biased low by O(sqrt(dt)) (-4.2% at dt = 0.01 on
    the toy, sim/results/benchmark_bias.json). Use `benchmark_hgi_e` (sqrt(dt)-extrapolated, both legs on one
    Brownian path) for any quoted number.
    """
    rng = np.random.default_rng() if rng is None else rng
    L = np.linalg.cholesky(rsn.Sigma)
    steps = int(T / dt)
    burn = int(burn_frac * steps)
    thin = max(1, (steps - burn) // 5000) if thin is None else thin

    w = np.zeros(rsn.I)
    sq = np.sqrt(dt)
    kept = []
    for k in range(steps):
        w = np.maximum(w + rsn.theta * dt + L @ (sq * rng.standard_normal(rsn.I)), 0.0)
        if k >= burn and (k - burn) % thin == 0:
            kept.append(w.copy())
    vals = hhat_batch(np.asarray(kept), rsn)
    return float(vals.mean()), np.asarray(kept)


# ---------------------------------------------------------------------------
# Uniform-snapshot Monte Carlo estimator of J_E (multi-path version)
# ---------------------------------------------------------------------------
def estimate_JE_uniform(rsn, pol_factory, r, tau=20.0, n_paths=4,
                        n_snap_per_path=500, burn_frac=0.3, rng=None):
    """
    J_E estimate from uniform time snapshots on n_paths INDEPENDENT paths
    (fixes the correlated-single-path SE problem: the SE is computed across
    path means, which are independent).
    """
    rng = np.random.default_rng() if rng is None else rng
    means = []
    for _ in range(n_paths):
        out = gillespie_ergodic(rsn, pol_factory(), r, T_diff=tau,
                                burn_frac=burn_frac,
                                n_snapshots=n_snap_per_path, rng=rng)
        means.append(out["JE"])
    means = np.asarray(means)
    se = means.std(ddof=1) / np.sqrt(len(means)) if len(means) > 1 else float("nan")
    return float(means.mean()), float(se), means


# ---------------------------------------------------------------------------
# High-precision ergodic HGI benchmark (exact componentwise reflection map)
# ---------------------------------------------------------------------------
def _rbm_mean_hhat(rsn, hhat_fn, dt, T, seed, n_lp=5000, chunk=10_000_000):
    """Time average of hhat_fn(W) for one RBM path at step dt (exact Skorokhod map)."""
    L = np.linalg.cholesky(rsn.Sigma)
    rng = np.random.default_rng(seed)
    steps = int(T / dt)
    thin = 1 if getattr(hhat_fn, "method", "vertices") != "lp" else max(1, steps // n_lp)
    A_level = np.zeros(rsn.I)
    run_min = np.zeros(rsn.I)
    acc, n, done = 0.0, 0, 0
    kept = []
    while done < steps:
        m = min(chunk, steps - done)
        dX = rng.standard_normal((m, rsn.I)) @ L.T * np.sqrt(dt) + rsn.theta * dt
        A = A_level + np.cumsum(dX, axis=0)
        Mn = np.minimum(np.minimum.accumulate(A, axis=0), run_min)
        W = A - Mn
        if thin == 1:
            acc += float(np.sum(hhat_fn(W))); n += m
        else:
            idx = np.arange((-done) % thin, m, thin)
            kept.append(W[idx])
        A_level, run_min = A[-1], Mn[-1]
        done += m
    if thin == 1:
        return acc / n
    return float(np.mean(hhat_fn(np.vstack(kept))))


def _rbm_mean_hhat_pair(rsn, hhat_fn, dt, k, T, seed, n_lp=5000, chunk=10_000_000):
    """Time averages of hhat_fn(W) for ONE Brownian path sampled at step dt and at step k*dt (the coarse
    increments are the fine ones summed in blocks of k), each reflected by the per-coordinate Skorokhod map
    of its own grid. Returns (fine, coarse). The random numbers are consumed exactly as in
    `_rbm_mean_hhat(rsn, hhat_fn, dt, T, seed)`, so `fine` equals that function's value bit for bit."""
    L = np.linalg.cholesky(rsn.Sigma)
    rng = np.random.default_rng(seed)
    steps = int(T / dt)
    lp = getattr(hhat_fn, "method", "vertices") == "lp"
    thin = (max(1, steps // n_lp), max(1, steps // k // n_lp)) if lp else (1, 1)
    chunk -= chunk % k
    st = [dict(A=np.zeros(rsn.I), mn=np.zeros(rsn.I), acc=0.0, n=0, kept=[], done=0) for _ in range(2)]
    done = 0
    while done < steps:
        m = min(chunk, steps - done)
        dXf = rng.standard_normal((m, rsn.I)) @ L.T * np.sqrt(dt) + rsn.theta * dt
        mc = m // k                                           # a ragged tail (< k fine steps) is dropped from the coarse leg
        for lev, dX in enumerate((dXf, dXf[:mc * k].reshape(mc, k, rsn.I).sum(axis=1))):
            if len(dX) == 0:
                continue
            q = st[lev]
            A = q["A"] + np.cumsum(dX, axis=0)
            Mn = np.minimum(np.minimum.accumulate(A, axis=0), q["mn"])
            W = A - Mn
            if thin[lev] == 1:
                q["acc"] += float(np.sum(hhat_fn(W))); q["n"] += len(dX)
            else:
                q["kept"].append(W[np.arange((-q["done"]) % thin[lev], len(dX), thin[lev])])
            q["A"], q["mn"] = A[-1], Mn[-1]
            q["done"] += len(dX)
        done += m
    if not lp:
        return st[0]["acc"] / st[0]["n"], st[1]["acc"] / st[1]["n"]
    return tuple(float(np.mean(hhat_fn(np.vstack(q["kept"])))) for q in st)


def benchmark_hgi_e(rsn, hhat_fn=None, dt=(4e-3, 1e-3), T=20000.0, seeds=(0, 1, 2, 3, 4, 5, 6, 7),
                    n_lp=5000, return_details=False):
    """
    HGI_E = E_pi[hhat(W)] for the limit RBM dW = theta dt + Sigma^{1/2} dB reflected
    componentwise in R^I_+, by the per-coordinate Skorokhod map on a discretized
    Brownian path (vectorized running minima). On a grid this map is the same scheme
    as reflected Euler (the two agree to rounding error, sim/results/benchmark_bias.json);
    it misses the minima between grid points, so the time average is biased low by
    O(sqrt(dt)): -4.2% at dt = 1e-2, -1.3% at dt = 1e-3 on the toy.

    With `dt` a pair (dt_1 > dt_2, dt_1/dt_2 an integer) the estimate is
    sqrt(dt)-extrapolated seed by seed,
        e = e_2 + (e_2 - e_1) sqrt(dt_2)/(sqrt(dt_1) - sqrt(dt_2)),
    with BOTH legs evaluated on the SAME Brownian path (one path per seed at step dt_2;
    the dt_1 leg sums its increments in blocks), so the path noise cancels in e_2 - e_1
    and the SE of e is that of a single leg. (mean, SE) are taken across seeds. A scalar
    `dt` gives the plain fixed-step estimate. The variant that draws
    the two legs from different paths is `benchmark_hgi_e_independent_legs`.

    hhat_fn: vectorized ideal cost (n x I -> n); default is the exact
    dual-vertex formula of `make_hhat` (LP fallback thins to n_lp points).
    Toy (theta = -1, Sigma = [[4,2],[2,4]]), defaults (8 seeds): see
    sim/results/multi_bench_crn.json. Check: networks B and C have hhat(w) = w_1 + w_2,
    so HGI_E = sum_i Sigma_ii / (2 |theta_i|) = 4 exactly.
    """
    hhat_fn = make_hhat(rsn) if hhat_fn is None else hhat_fn
    if np.isscalar(dt):
        vals = np.array([_rbm_mean_hhat(rsn, hhat_fn, float(dt), T, s, n_lp) for s in seeds])
        per_dt = {float(dt): vals}
    else:
        dt1, dt2 = (float(d) for d in dt)
        if not dt1 > dt2 > 0:
            raise ValueError("dt must be a pair (dt_1 > dt_2 > 0) or a scalar")
        ratio = dt1 / dt2
        if abs(ratio - round(ratio)) > 1e-9:
            raise ValueError("dt_1 / dt_2 must be an integer (both legs use the same Brownian path)")
        legs = np.array([_rbm_mean_hhat_pair(rsn, hhat_fn, dt2, int(round(ratio)), T, s, n_lp) for s in seeds])
        e2, e1 = legs[:, 0], legs[:, 1]
        s1, s2 = math.sqrt(dt1), math.sqrt(dt2)
        vals = e2 + (e2 - e1) * s2 / (s1 - s2)
        per_dt = {dt1: e1, dt2: e2}
    mean = float(vals.mean())
    se = float(vals.std(ddof=1) / np.sqrt(len(vals))) if len(vals) > 1 else float("nan")
    if return_details:
        return mean, se, dict(per_seed=vals, per_dt=per_dt, seeds=tuple(seeds), T=T)
    return mean, se


def benchmark_hgi_e_independent_legs(rsn, hhat_fn=None, dt=(4e-3, 1e-3), T=20000.0, seeds=(0, 1, 2, 3),
                    n_lp=5000, return_details=False):
    """
    Benchmark with independent extrapolation legs (it produced the HGI_E values of
    sim/results/multi_bench.json, toy 2.81 +- 0.02). Use `benchmark_hgi_e` for any quoted number.
    Its two legs are drawn from DIFFERENT Brownian paths (the dt_1 leg and the dt_2 leg consume the seed's
    stream at different step sizes), so the extrapolation 2 e_2 - e_1 carries the noise of both legs: on
    networks B and C, where HGI_E = 4 exactly, it returned 4.069 +- 0.026.

    HGI_E = E_pi[hhat(W)] for the limit RBM dW = theta dt + Sigma^{1/2} dB reflected
    componentwise in R^I_+, by the per-coordinate Skorokhod map on a
    discretized Brownian path (vectorized running minima).

    The remaining discretization error of the time average is O(sqrt(dt)), so
    with `dt` a pair (dt_1 > dt_2) the estimate is sqrt(dt)-extrapolated
    seed by seed:  e = e_2 + (e_2 - e_1) sqrt(dt_2)/(sqrt(dt_1) - sqrt(dt_2)),
    and (mean, SE) are taken across seeds. A scalar `dt` gives the plain
    fixed-step estimate (biased low by ~2% at dt = 1e-3 on the toy).

    hhat_fn: vectorized ideal cost (n x I -> n); default is the exact
    dual-vertex formula of `make_hhat` (LP fallback thins to n_lp points).
    Toy (theta = -1, Sigma = [[4,2],[2,4]]): 2.81 +- 0.02 with the defaults.
    """
    hhat_fn = make_hhat(rsn) if hhat_fn is None else hhat_fn
    if np.isscalar(dt):
        vals = np.array([_rbm_mean_hhat(rsn, hhat_fn, float(dt), T, s, n_lp) for s in seeds])
        per_dt = {float(dt): vals}
    else:
        dt1, dt2 = (float(d) for d in dt)
        if not dt1 > dt2 > 0:
            raise ValueError("dt must be a pair (dt_1 > dt_2 > 0) or a scalar")
        e1 = np.array([_rbm_mean_hhat(rsn, hhat_fn, dt1, T, s, n_lp) for s in seeds])
        e2 = np.array([_rbm_mean_hhat(rsn, hhat_fn, dt2, T, s, n_lp) for s in seeds])
        s1, s2 = math.sqrt(dt1), math.sqrt(dt2)
        vals = e2 + (e2 - e1) * s2 / (s1 - s2)
        per_dt = {dt1: e1, dt2: e2}
    mean = float(vals.mean())
    se = float(vals.std(ddof=1) / np.sqrt(len(vals))) if len(vals) > 1 else float("nan")
    if return_details:
        return mean, se, dict(per_seed=vals, per_dt=per_dt, seeds=tuple(seeds), T=T)
    return mean, se


# ---------------------------------------------------------------------------
# Discounted HGI benchmark (limit process, from a given initial workload)
# ---------------------------------------------------------------------------
def _rbm_discounted_hhat(rsn, hhat_fn, w0, varsigma, dt, T, n_paths, seed, chunk=2000, coarse=None):
    """Mean over paths of int_0^T e^{-varsigma t} h_hat(W(t)) dt (trapezoid on the dt-grid) for the RBM
    dW = theta dt + Sigma^{1/2} dB started at w0, reflected componentwise by the exact 1-d Skorokhod map
    of the discretized path. Returns (mean, SE over paths); with `coarse` = k (integer) it also evaluates the
    same Brownian paths on the k-times coarser grid (common random numbers for the sqrt(dt) extrapolation) and
    returns (mean_fine, se_fine, mean_coarse, se_coarse, se_of_difference)."""
    L = np.linalg.cholesky(rsn.Sigma)
    n = int(round(T / dt))
    if coarse is not None and n % coarse:
        raise ValueError("T/dt must be a multiple of the coarsening factor")

    def weights(step, npts):
        tg = np.arange(npts + 1) * step
        w = np.exp(-varsigma * tg) * step
        w[0] *= 0.5; w[-1] *= 0.5
        return w

    def reflect_cost(dX, k):
        X = w0 + np.concatenate([np.zeros((k, 1, rsn.I)), np.cumsum(dX, axis=1)], axis=1)
        W = X - np.minimum(np.minimum.accumulate(X, axis=1), 0.0)
        return hhat_fn(W.reshape(-1, rsn.I)).reshape(k, -1)

    wf = weights(dt, n)
    wc = weights(dt * coarse, n // coarse) if coarse else None
    rng = np.random.default_rng(seed)
    fine, crs = [], []
    for c0 in range(0, n_paths, chunk):
        k = min(chunk, n_paths - c0)
        dX = rng.standard_normal((k, n, rsn.I)) @ L.T * math.sqrt(dt) + rsn.theta * dt
        fine.append(reflect_cost(dX, k) @ wf)
        if coarse:
            crs.append(reflect_cost(dX.reshape(k, n // coarse, coarse, rsn.I).sum(axis=2), k) @ wc)
    f = np.concatenate(fine)
    if not coarse:
        return float(f.mean()), float(f.std(ddof=1) / math.sqrt(len(f)))
    c = np.concatenate(crs); d = f - c
    return (float(f.mean()), float(f.std(ddof=1) / math.sqrt(len(f))), float(c.mean()),
            float(c.std(ddof=1) / math.sqrt(len(c))), float(d.std(ddof=1) / math.sqrt(len(d))))


def benchmark_hgi_d(rsn, w0=None, varsigma=1.0, hhat_fn=None, dt=(1e-3, 5e-4), T=20.0, n_paths=10000,
                    seed=12345, return_details=False):
    """
    HGI_D(w0) = int_0^oo e^{-varsigma t} E h_hat(W(t)) dt  (paper eq. (11), unnormalized: no factor varsigma)
    for the limit RBM started at w0 (default 0, the empty system), by the exact per-coordinate Skorokhod map
    on a discretized Brownian path, truncated at T (tail e^{-varsigma T} E h_hat ~ 1e-8 at the defaults).

    As for the ergodic benchmark the fixed-dt estimate is biased low by O(sqrt(dt)); with `dt` a pair
    (dt_1 > dt_2, dt_1/dt_2 an integer) the result is sqrt(dt)-extrapolated,
    e = e_2 + (e_2 - e_1) sqrt(dt_2)/(sqrt(dt_1) - sqrt(dt_2)), with BOTH legs evaluated on the same Brownian
    paths (the coarse path is the fine one summed in blocks), so the extrapolation term's noise is the SE of the
    paired difference, not 2.4x the leg SE. A scalar `dt` gives the plain estimate.
    Toy, w0 = 0, varsigma = 1, defaults: see sim/results/hgi_d_toy.json (recorded value 1.33).
    Note: every queueing-side J_D in this project uses varsigma = 1 (toykern3.c hard-codes it); this routine is
    the only place varsigma != 1 can be evaluated.
    """
    hhat_fn = make_hhat(rsn) if hhat_fn is None else hhat_fn
    w0 = np.zeros(rsn.I) if w0 is None else np.asarray(w0, float).reshape(rsn.I)
    if np.any(w0 < 0) or varsigma <= 0:
        raise ValueError("w0 must be >= 0 and varsigma > 0")
    if np.isscalar(dt):
        m, se = _rbm_discounted_hhat(rsn, hhat_fn, w0, varsigma, float(dt), T, n_paths, seed)
        det = {float(dt): (m, se)}
    else:
        dt1, dt2 = (float(d) for d in dt)
        if not dt1 > dt2 > 0:
            raise ValueError("dt must be a pair (dt_1 > dt_2 > 0) or a scalar")
        ratio = dt1 / dt2
        if abs(ratio - round(ratio)) > 1e-9:
            raise ValueError("dt_1 / dt_2 must be an integer (common random numbers across the two legs)")
        e2, s2e, e1, s1e, sde = _rbm_discounted_hhat(rsn, hhat_fn, w0, varsigma, dt2, T, n_paths, seed,
                                                     coarse=int(round(ratio)))
        s1, s2 = math.sqrt(dt1), math.sqrt(dt2)
        f = s2 / (s1 - s2)
        m = e2 + (e2 - e1) * f
        se = math.sqrt(s2e ** 2 + (f * sde) ** 2)
        det = {dt1: (e1, s1e), dt2: (e2, s2e), "se_difference": sde}
    if return_details:
        return m, se, dict(per_dt=det, varsigma=varsigma, w0=w0.tolist(), T=T, n_paths=n_paths)
    return m, se

### `je.py` — run_JE: ergodic cost with CIs on simulation data

In [ ]:
%%writefile je.py
# Standalone module with plain imports; pure Python. The compiled kernels are in sim/.

"""
je.py -- estimate the ergodic cost J_E (and the SSC gap) on simulation data.

Built on exact Gillespie simulation, time-weighted averages and batch-means
confidence intervals.

    from rsn import make_2lln
    from je import run_JE
    out = run_JE(make_2lln(), r=50, T=10)
    print(out["JE"], out["JE_ci"], out["ssc_gap"])
"""
import numpy as np

from offline import OfflineTables
from policy import GeneralThresholdPolicy
from simulation import gillespie_ergodic
from je_stats import batch_means_ci


def run_JE(rsn, r, T=10.0, c1=1.0, c2=4.0, kappa=0.15, seed=0,
           n_snapshots=400, burn_frac=0.3, E_mask=None, vb_norm=None,
           gamma_c=0.98, gamma_b=0.98, hhat_fn=None, n_batches=16,
           tables=None):
    """
    One ergodic run of the threshold policy on the network `rsn` at
    heavy-traffic scale r over T diffusion time units (pure-Python exact
    Gillespie). Defaults instantiate Definition 2.3 verbatim. E_mask=0 (flags
    off) and vb_norm above b_max are the CONTROL EXPERIMENT of the report:
    labeled departures OUTSIDE Definition 2.3, never a recommended setting.

    Returns dict(JE, JE_ci, ssc_gap, gap_ci, E_frac, snapshots).
    """
    tables = (OfflineTables(rsn, gamma_c=gamma_c, gamma_b=gamma_b,
                            vb_norm=vb_norm) if tables is None else tables)
    pol = GeneralThresholdPolicy(rsn, c1=c1, c2=c2, kappa=kappa,
                                 tables=tables, E_mask=E_mask)
    out = gillespie_ergodic(rsn, pol, float(r), T_diff=T,
                            burn_frac=burn_frac, n_snapshots=n_snapshots,
                            rng=np.random.default_rng(seed))
    snaps = out["snapshots"]
    je_series = (snaps @ rsn.h) / float(r)
    if hhat_fn is not None:
        hg = np.asarray(hhat_fn((snaps @ (rsn.K @ rsn.M).T) / float(r)))
    else:
        from simulation import hhat_batch
        hg = hhat_batch((snaps @ (rsn.K @ rsn.M).T) / float(r), rsn)
    ci_je = batch_means_ci(je_series, n_batches)
    ci_gap = batch_means_ci(je_series - hg, n_batches)
    return dict(JE=out["JE"], JE_ci=ci_je["half_width"],
                ssc_gap=float(np.mean(je_series - hg)),
                gap_ci=ci_gap["half_width"], E_frac=out["E_frac"],
                snapshots=snaps)

### `check.py` — end-to-end verification driver

In [ ]:
%%writefile check.py
# Standalone module with plain imports; pure Python. The compiled kernels are in sim/.

"""
check.py -- end-to-end verification driver.

Builds the paper toy (Example 2.2), runs the offline pass with all
admissibility assertions, estimates J_E and the SSC gap on simulation
data (je.py) under Definition 2.3 verbatim, computes the RBM benchmark
HGI_E (simulation.py, exact reflection map, sqrt(dt)-extrapolated), and
then runs the CONTROL EXPERIMENT of the report (hysteresis flags off,
||v^b|| = 0.24 > b_max) for comparison. The control experiment is OUTSIDE
Definition 2.3; it is printed as such and is not a recommended setting.
Runs in ~1 minute in Colab; raise r and T for report-grade numbers.

    python check.py
"""
import numpy as np

from rsn import make_2lln
from offline import OfflineTables
from je import run_JE
from simulation import benchmark_hgi_e

if __name__ == "__main__":
    net = make_2lln()
    hhat = lambda W: np.asarray(W).max(axis=1)   # analytic for the 2LLN

    print(net)
    tab = OfflineTables(net, gamma_c=0.98, gamma_b=0.98)
    print()
    print(tab.summary())

    print("\n-- Definition 2.3 verbatim (SSC gap is the finite-r test) --")
    out = run_JE(net, r=50, T=10, c1=1.0, c2=4.0, kappa=0.15, seed=1,
                 hhat_fn=hhat, tables=tab)
    print(f"J_E     = {out['JE']:.3f} +- {out['JE_ci']:.3f}")
    print(f"SSC gap = {out['ssc_gap']:.4f} +- {out['gap_ci']:.4f}"
          f"   (decays like c2*r^(kappa-1))")
    print(f"E-idle  = {out['E_frac']:.3f}")

    print("\n-- RBM benchmark HGI_E (exact reflection map, sqrt(dt)-extrapolated) --")
    bench, se = benchmark_hgi_e(net, hhat_fn=hhat, T=20000.0, seeds=(0, 1))
    print(f"HGI_E   = {bench:.3f} +- {se:.3f}   (2.81 +- 0.02 with 4 seeds)")

    print("\n-- CONTROL EXPERIMENT (outside Definition 2.3): E_mask=0, vb_norm=0.24 --")
    print("   flags disabled and ||v^b|| above b_max; a labeled departure, NOT a recommendation")
    out2 = run_JE(net, r=50, T=20, c1=1.0, c2=10.0, kappa=0.22, seed=1,
                  E_mask=0, vb_norm=0.24, hhat_fn=hhat)
    print(f"J_E     = {out2['JE']:.3f} +- {out2['JE_ci']:.3f}")
    ratio = out2["JE"] / bench
    print(f"control level / benchmark = {ratio:.2f}")
    verdict = "PASS" if ratio < 1.6 else "CHECK SETTINGS"
    print(f"RESULT: {verdict}")

## Part 2 — run each file on simulation data

In [ ]:
# run make_2lln from rsn.py — the network (edit K/alpha/beta/h for your own)
import numpy as np
from rsn import make_2lln, RSN
net = make_2lln()
hhat = lambda W: np.asarray(W).max(axis=1)    # analytic ideal cost for the 2LLN
print(net)

In [ ]:
# run OfflineTables from offline.py — the offline construction
from offline import OfflineTables
tab = OfflineTables(net, gamma_c=0.98, gamma_b=0.98)
print(tab.summary())

In [ ]:
# run GeneralThresholdPolicy from policy.py — one allocation decision
from policy import GeneralThresholdPolicy
pol = GeneralThresholdPolicy(net, c1=1.0, c2=4.0, kappa=0.15, tables=tab)
b, Z, E = pol.allocation(q=np.array([3.0, 8.0, 40.0]), r=100)
print(f"queue (3, 8, 40) at r=100  ->  b = {np.round(b, 3)}, Z = {Z}, E = {E.astype(int)}")

In [ ]:
# run gillespie_ergodic from simulation.py — exact simulation of the policy
from simulation import gillespie_ergodic
out = gillespie_ergodic(net, pol, r=50.0, T_diff=10.0, n_snapshots=400,
                        rng=np.random.default_rng(1))
print(f"r=50: J_E = {out['JE']:.3f}   SSC gap = {out['ssc_gap']:.4f}   "
      f"E-idle = {out['E_frac']:.3f}")

In [ ]:
# run run_JE from je.py on simulation data — ergodic cost with confidence intervals
from je import run_JE
res = run_JE(net, r=50, T=20, c1=1.0, c2=10.0, kappa=0.22, seed=1, hhat_fn=hhat)   # Definition 2.3 verbatim
print(f"J_E = {res['JE']:.3f} ± {res['JE_ci']:.3f}    "
      f"SSC gap = {res['ssc_gap']:.4f} ± {res['gap_ci']:.4f}")
# CONTROL EXPERIMENT (outside Definition 2.3): flags off, ||v^b|| = 0.24 > b_max. Labeled, not recommended.
ctrl = run_JE(net, r=50, T=20, c1=1.0, c2=10.0, kappa=0.22, seed=1,
              E_mask=0, vb_norm=0.24, hhat_fn=hhat)
print(f"control experiment (outside Def. 2.3): J_E = {ctrl['JE']:.3f} ± {ctrl['JE_ci']:.3f}")

In [ ]:
# run benchmark_hgi_e from simulation.py — the RBM benchmark (exact reflection map,
# sqrt(dt)-extrapolated from dt = 4e-3, 1e-3), then compare
from simulation import benchmark_hgi_e
bench, se = benchmark_hgi_e(net, hhat_fn=hhat, T=20000.0, seeds=(0, 1))   # 2.81 ± 0.02 with 4 seeds
print(f"HGI_E = {bench:.3f} ± {se:.3f}")
print(f"Def. 2.3 level / benchmark = {res['JE'] / bench:.2f};   control experiment / benchmark = {ctrl['JE'] / bench:.2f}")

In [ ]:
# run check.py — the full verification driver in one line
!python check.py

---
`check.py` is the file to edit for your own experiments: swap `make_2lln()`
for your own `RSN(K=..., alpha=..., beta=..., h=..., eta=...)` (or
`alpha_bar=..., beta_bar=...` for Condition 2 in the paper's form), raise `r`
and `T`, and re-run. Definition 2.3 verbatim is the default everywhere;
`E_mask=0` / `vb_norm=…` is the **control experiment (outside Definition 2.3)**
from the report — a labeled departure, never a recommended configuration.

## Part 3 — the compiled engine (r = 1000 in under a minute)

`sim/colab_engine.py` compiles `sim/genkern.c` with Colab's gcc at import (~2 s) and
exposes `Net(name, K, alpha, beta, h, eta)` with `run(r, T, c1, c2, kappa, seed)`;
1.3e7 events/s, i.e. the toy at r = 1000, T = 40 (3e8 events) in ~25 s. It needs the
workspace folder (`sim/` and `rsn_flat/` side by side): clone it, or upload
`rsn_hgi_workspace.zip` to Drive and unzip it. If compilation fails it falls back to the
pure-Python loop (then keep r ≤ 100). Everything is Definition 2.3 verbatim.

In [ ]:
# --- one r = 1000 toy run with the compiled engine (same-path HGI, flags, stationarity check) ---
import os, sys, glob, time
WS = '/content/rsn_hgi_workspace'                       # <-- where the workspace folder lives
if not os.path.isdir(WS):                               # e.g. unzipped in Drive instead
    hits = glob.glob('/content/drive/MyDrive/**/colab_engine.py', recursive=True)
    assert hits, 'put the workspace at /content/rsn_hgi_workspace or in Drive first'
    WS = os.path.dirname(os.path.dirname(hits[0]))
sys.path.insert(0, os.path.join(WS, 'sim'))
import colab_engine as ce                                # compiles genkern.c here
print('engine:', ce.ENGINE, '| kernel:', ce.SO_PATH)

nw = ce.Net('toy', [[1, 0, 1], [0, 1, 1]], [1, 1, 1], [1, 1, 1], [1, 1, 1], [-0.5] * 3)
t0 = time.time()
o = nw.run(1000, 40, 0.1, 60, 0.24, seed=1)              # deep band (0.1, 60, 0.24): Definition 2.3 verbatim
print(f"{time.time() - t0:.0f} s, {o['events']:.2e} events")
print(f"J_E = {o['JE']:.3f}   same-path HGI = {o['hgi_path']:.3f}   excess = {o['JE'] - o['hgi_path']:.3f}"
      f" ({100 * (o['JE'] - o['hgi_path']) / 2.81:.0f}% of HGI_E = 2.81; expect ~31% at r = 1000)")
print(f"E-fraction = {o['E_frac']:.1e}   halves {o['JE_1st']:.2f} -> {o['JE_2nd']:.2f} (growth would mean transient)")